# Proyecto Final Minería - Semana 2
## Transformación de Conjunto de Datos


En base al Análisis Exploratorio de Datos (EDA) realizado en la Semana 1, identificamos los siguientes problemas y áreas de oportunidad para la transformación:
1. Existen columnas constantes que no aportan información al modelo (`indicator`, `unit`, `source_id`).
2. La columna `notes_ids` contiene un 100% de valores nulos.
3. Existen registros con un grupo etario igual a `Total`, lo cual es un valor agregado y puede causar ruido o sobreajuste al modelo si se analiza junto con los subgrupos.
4. Las variables categóricas necesitan ser transformadas a un formato numérico para los algoritmos de Machine Learning (Codificación ordinal para edades y One-Hot Encoding para países).

In [1]:
import pandas as pd

# 1. Cargar el conjunto de datos original
df = pd.read_csv("data_1777144519.csv", sep=";")

print("Dimensiones originales:", df.shape)
df.head()

Dimensiones originales: (870, 8)


,indicator,País__ESTANDAR,Grupos etarios Uso Internet,Años__ESTANDAR,value,unit,notes_ids,source_id
0,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2016,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
1,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2017,76,Porcentaje sobre el total de personas en cada ...,NaN,9353
2,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2018,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
3,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2019,79,Porcentaje sobre el total de personas en cada ...,NaN,9353
4,Personas usuarias de Internet por grupo etario...,Argentina,edad de medicion a 17 años,2020,88,Porcentaje sobre el total de personas en cada ...,NaN,9353


### Paso 1: Manejo de valores agrupados (Remover 'Total')
**Justificación:** El grupo etario `Total` es una agregación matemática de las otras edades. Mantenerlo en el conjunto de datos podría causar multicolinealidad y sesgar los modelos predictivos que asumen que las filas son observaciones independientes.

In [2]:
# Filtrar las filas donde el grupo etario es 'Total'
df_transformado = df[df['Grupos etarios Uso Internet'] != 'Total'].copy()

print("Dimensiones después de eliminar 'Total':", df_transformado.shape)

Dimensiones después de eliminar 'Total': (725, 8)


### Paso 2: Eliminación de Columnas Irrelevantes y Manejo de Nulos
**Justificación:** 
- `notes_ids`: Tiene el 100% de sus valores faltantes (nulos), por lo que no es recuperable ni útil.
- `indicator`, `unit`, `source_id`: Tienen cardinalidad igual a 1 (varianza 0). Al ser valores constantes, no tienen poder predictivo ni ayudan a discriminar o encontrar patrones.

In [3]:
# Eliminar columnas constantes y con valores faltantes
columnas_a_eliminar = ['indicator', 'unit', 'notes_ids', 'source_id']
df_transformado = df_transformado.drop(columns=columnas_a_eliminar)

print("Columnas restantes:", df_transformado.columns.tolist())

Columnas restantes: ['País__ESTANDAR', 'Grupos etarios Uso Internet', 'Años__ESTANDAR', 'value']


### Paso 3: Codificación Ordinal (Grupos Etarios)
**Justificación:** La variable `Grupos etarios Uso Internet` tiene un orden inherente (de menor a mayor edad). La codificación ordinal preserva esta relación lógica y jerárquica para los modelos predictivos.

In [4]:
# Mapeo ordinal de edades
mapeo_edades = {
    'edad de medicion a 17 años': 0,
    '18 a 25 años de edad': 1,
    '26 a 50 años de edad': 2,
    '51 a 65 años': 3,
    '66 años en adelante': 4
}

df_transformado['grupo_etario_ord'] = df_transformado['Grupos etarios Uso Internet'].map(mapeo_edades)

df_transformado[['Grupos etarios Uso Internet', 'grupo_etario_ord']].head()

,Grupos etarios Uso Internet,grupo_etario_ord
0,edad de medicion a 17 años,0
1,edad de medicion a 17 años,0
2,edad de medicion a 17 años,0
3,edad de medicion a 17 años,0
4,edad de medicion a 17 años,0


### Paso 4: One-Hot Encoding (Países)
**Justificación:** La variable `País__ESTANDAR` es categórica nominal (no hay un orden natural entre países). Usar One-Hot Encoding es la mejor práctica para que los modelos no asuman relaciones de proximidad numérica incorrectas.

In [5]:
# Aplicar One-Hot Encoding a la columna de países
df_transformado = pd.get_dummies(df_transformado, columns=['País__ESTANDAR'], dtype=int)

print("Dimensiones tras One-Hot Encoding:", df_transformado.shape)
df_transformado.head()

Dimensiones tras One-Hot Encoding: (725, 18)


,Grupos etarios Uso Internet,Años__ESTANDAR,value,grupo_etario_ord,País__ESTANDAR_Argentina,País__ESTANDAR_Bolivia (Estado Plurinacional de),País__ESTANDAR_Brasil,País__ESTANDAR_Chile,País__ESTANDAR_Colombia,País__ESTANDAR_Costa Rica,País__ESTANDAR_Ecuador,País__ESTANDAR_El Salvador,País__ESTANDAR_Honduras,País__ESTANDAR_México,País__ESTANDAR_Panamá,País__ESTANDAR_Paraguay,País__ESTANDAR_Perú,País__ESTANDAR_Uruguay
0,edad de medicion a 17 años,2016,76,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,edad de medicion a 17 años,2017,76,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,edad de medicion a 17 años,2018,79,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,edad de medicion a 17 años,2019,79,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
4,edad de medicion a 17 años,2020,88,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


### Paso 5: Guardar el Conjunto de Datos Transformado
**Justificación:** Guardamos el dataset final en un formato limpio (CSV) listo para ser ingerido por los modelos de Machine Learning en las siguientes fases del proyecto.

In [6]:
# Guardar el conjunto de datos resultante a un archivo CSV
ruta_guardado = 'datos_transformados.csv'
df_transformado.to_csv(ruta_guardado, index=False)

print(f"Conjunto de datos transformado guardado exitosamente en: {ruta_guardado}")

Conjunto de datos transformado guardado exitosamente en: datos_transformados.csv
